# Nuclear scaling — analysis

Everything is read from `nuclear_scaling.db` and its radial-profile parquet
sidecars. No CSVs. Adding a replicate is one `build_db.py import` plus a re-run
of this notebook, unchanged.

**Kernel: `starforge`** — the TensorFlow env has no pyarrow and cannot read the
parquet.

All plotting code lives in these cells so you can edit it directly. Only
`nsdb.py` is imported as a module.

### Structure

Every figure is a time series. Four tracked nuclei (`FOCAL`) are followed through
each analysis so individual trajectories stay visible next to the population
summary — the question is how a nucleus changes, not what the population looked
like once.

`Nucleus_ID` in the export is the **track id**, so a nucleus keeps its identity
across frames and `FOCAL` selects the four longest-lived tracks.

### Known limits of the v18.1 export

| Limitation | Consequence |
|---|---|
| `Background_*`, `Halo*_NPC`, `Halo*_Membrane` empty | `nc_ratio` has no background subtraction — read the trend, not the level |
| `Rho_Normalized` is centre→wall | radial position is in ρ, not µm; `rho_wall` is reconstructed as a proxy |
| `Distance_px` not exported | no absolute radii anywhere |
| repair not written back to `grouped_z_df` | `nucleus_z_stack` areas are pre-repair — use `nuclei` for area |
| `Droplet_ID` empty | droplet unavailable as a grouping level |


## 1 · Setup

In [ ]:
from pathlib import Path
import os, sys, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)


# Project layout changed 2026-09-04: modules moved to src/, the database to
# data/db/, derived data to data/derived/, figures to outputs/figures/.
# This is a permanent fix to the paths, not a one-off patch -- the marker is now
# "src/nsdb.py" and src/ (not the project root) is what goes on sys.path.
def find_project_root(marker="src/nsdb.py", start=None):
    """Anchor on a file so this runs from any machine or subdirectory."""
    here = (start or Path.cwd()).resolve()
    for d in (here, *here.parents):
        if (d / marker).exists():
            return d
    raise FileNotFoundError(f"no {marker} in {here} or any parent")


PROJECT = find_project_root()
SRC     = PROJECT / "src"
DB      = PROJECT / "data" / "db" / "nuclear_scaling.db"
DERIVED = PROJECT / "data" / "derived"
FIGDIR  = PROJECT / "outputs" / "figures"; FIGDIR.mkdir(parents=True, exist_ok=True)

os.environ["NUCLEAR_SCALING_DB"]   = str(DB)
os.environ["NUCLEAR_SCALING_ROOT"] = str(DERIVED)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import nsdb

pd.set_option("display.max_columns", 60, "display.width", 180)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10})

EXPERIMENT   = "control_extract_1.1"
N_FOCAL      = 4
BIN_MIN      = 6.0
SHELLS       = [(0.00, 0.25), (0.25, 0.50), (0.50, 0.75), (0.75, 1.00)]
SHELL_COLORS = ["#7fd4b0", "#2fa383", "#4c6fbf", "#1a1a2e"]
FOCAL_COLORS = ["#c1121f", "#0a6f8a", "#e08214", "#5b3a91"]

print("python  :", sys.executable)
print("project :", PROJECT)
print("database:", DB.name, f"({DB.stat().st_size/1e6:.1f} MB)")

## 2 · Inventory and integrity

In [ ]:
display(nsdb.experiments()[["experiment_id", "experimental_group", "treatment",
                            "concentration", "biological_replicate", "experiment_date"]])
report = nsdb.audit()
display(report)
print("open issues:", *report.loc[~report.ok, "check"].tolist(), sep="\n  ")

## 3 · Shared analysis helpers

The only reusable code in the notebook. Everything below is plotting you can
edit in place.

**Angle convention.** The sweep casts rays as `y = cy + r·sin θ` and image y
increases downward, so θ runs *clockwise* on screen with θ=0 pointing right.
`orient_polar` matches that.

In [ ]:
def orient_polar(ax):
    """theta=0 at +x, increasing clockwise — matches the displayed image."""
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(-1)


def boxplot(ax, data, labels, **kw):
    """boxplot with whichever tick-label kwarg this matplotlib accepts.

    `labels=` became `tick_labels=` in matplotlib 3.9 and was later removed.
    """
    try:
        return ax.boxplot(data, tick_labels=labels, **kw)
    except TypeError:
        return ax.boxplot(data, labels=labels, **kw)


def short(nid):
    """'control_extract_1.1|FOV01|N000082' -> 'N000082'."""
    return str(nid).split("|")[-1]


def resultant(theta_rad, weight):
    """Weighted circular resultant -> (R, phi). R=0 isotropic, R=1 one direction."""
    w = np.asarray(weight, dtype=float)
    ok = np.isfinite(w) & np.isfinite(theta_rad) & (w > 0)
    if ok.sum() == 0 or w[ok].sum() <= 0:
        return np.nan, np.nan
    z = np.sum(w[ok] * np.exp(1j * theta_rad[ok])) / w[ok].sum()
    return float(abs(z)), float(np.angle(z))


def angular_profile(df, n_bins=36):
    """Bin one nucleus/frame/shell by angle -> (theta_rad, mean_intensity, n_samples).

    NaN where no ray sampled that wedge — that happens where the droplet wall
    clips the sweep, which is exactly why R_geom below is needed.
    """
    edges = np.linspace(0, 360, n_bins + 1)
    idx = pd.cut(df["theta_deg"] % 360, edges, labels=False, include_lowest=True)
    g = df.assign(_b=idx).groupby("_b")["intensity"]
    mean = g.mean().reindex(range(n_bins)).to_numpy(dtype=float)
    count = g.size().reindex(range(n_bins)).fillna(0).to_numpy(dtype=float)
    return np.radians(0.5 * (edges[:-1] + edges[1:])), mean, count


def add_rho_wall_proxy(sweep):
    """Reconstruct a surface-referenced radius: 0 at the nuclear surface, 1 at the wall.

    The pipeline's rho_wall needs distance_px and inside_nucleus, neither of
    which is exported. Per ray, the intensity maximum marks the NE/membrane
    ring, so it stands in for the nuclear surface:

        rho_wall_proxy = (rho - rho_peak) / (1 - rho_peak),  clipped to [0, 1]

    PROXY, not the real thing. The true surface is the last sample inside the
    nucleus mask, which sits slightly INSIDE the ring peak, so this reads a
    little large. Interior samples all clip to 0 — filter with `> 0` to keep
    the perinuclear region only.
    """
    key = ["nucleus_id", "time_frame", "theta_deg"]
    peak = (sweep.loc[sweep.groupby(key)["intensity"].idxmax(), key + ["rho_normalized"]]
            .rename(columns={"rho_normalized": "rho_peak"}))
    out = sweep.merge(peak, on=key, how="left")
    out["rho_wall_proxy"] = ((out.rho_normalized - out.rho_peak)
                             / (1 - out.rho_peak)).clip(0, 1)
    return out


def ray_peaks(sweep):
    """Per ray: the rho at which intensity peaks (the pipeline plots this in µm)."""
    idx = sweep.groupby(["nucleus_id", "time_frame", "theta_deg"])["intensity"].idxmax()
    return (sweep.loc[idx, ["nucleus_id", "time_frame", "theta_deg",
                            "rho_normalized", "intensity"]]
            .rename(columns={"rho_normalized": "rho_at_peak",
                             "intensity": "peak_intensity"}))


def asymmetry(sweep, shells=SHELLS, n_bins=36, baseline_pct=10.0,
              radius_col="rho_normalized"):
    """Per (nucleus, frame, shell) membrane asymmetry.

    R       intensity-weighted resultant after subtracting a per-shell baseline
    phi     its direction (radians, image convention)
    R_geom  resultant of the SAMPLE COUNTS alone — the apparent asymmetry an
            isotropic nucleus would show from droplet-wall clipping. R has to
            beat R_geom to mean anything.
    p       Rayleigh approximation exp(-n_eff*R^2), where n_eff is the effective
            weight count, so a profile carried by two bright bins is not
            credited with the full bin count.
    """
    rows = []
    for (nid, t), g in sweep.groupby(["nucleus_id", "time_frame"], sort=False):
        for lo, hi in shells:
            sub = g[(g[radius_col] > lo) & (g[radius_col] <= hi)]
            if sub.empty:
                continue
            th, mean, count = angular_profile(sub, n_bins)
            filled = np.isfinite(mean)
            if filled.sum() < 3:
                continue
            w = np.clip(mean - np.nanpercentile(mean[filled], baseline_pct), 0, None)
            w[~filled] = 0.0
            R, phi = resultant(th, w)
            Rg, _ = resultant(th, count)
            n_eff = (w.sum() ** 2 / np.sum(w ** 2)) if np.sum(w ** 2) > 0 else 0.0
            rows.append(dict(nucleus_id=nid, time_frame=t, shell=f"({lo:.2f}, {hi:.2f}]",
                             shell_lo=lo, shell_hi=hi, R=R, phi=phi, R_geom=Rg,
                             n_eff=n_eff, n_samples=len(sub),
                             p=float(np.exp(-n_eff * R ** 2)) if R == R else np.nan))
    return pd.DataFrame(rows)


def population_direction(asym):
    """Do nuclei agree on a lab-frame direction? Each nucleus gets unit weight."""
    rows = []
    for (shell, t), g in asym.groupby(["shell", "time_frame"]):
        phi = g["phi"].dropna().to_numpy()
        if phi.size == 0:
            continue
        R, mean_phi = resultant(phi, np.ones_like(phi))
        rows.append(dict(shell=shell, time_frame=t, n=phi.size, R_pop=R,
                         mean_phi=mean_phi, p=float(np.exp(-phi.size * R ** 2))))
    return pd.DataFrame(rows)


def sweep_grid(df, rho_bins=60, radius_col="rho_normalized", agg="mean", close_theta=True):
    """Bin a long-form sweep into a regular theta x rho grid for plot_surface.

    Each ray has a different sample count, so rho must be binned before the
    pivot. theta is closed by repeating the first ray at +360 so there is no seam.
    """
    edges = np.linspace(0, 1, rho_bins + 1)
    d = df[df[radius_col].between(0, 1)].copy()
    d["rb"] = pd.cut(d[radius_col], edges, labels=False, include_lowest=True)
    grid = (d.pivot_table(index="theta_deg", columns="rb", values="intensity", aggfunc=agg)
            .reindex(columns=range(rho_bins)))
    theta, z = grid.index.to_numpy(dtype=float), grid.to_numpy(dtype=float)
    if close_theta and theta.size > 1:
        theta = np.append(theta, theta[0] + 360.0)
        z = np.vstack([z, z[0]])
    rho = 0.5 * (edges[:-1] + edges[1:])
    THETA, RHO = np.meshgrid(theta, rho, indexing="ij")
    return THETA, RHO, z


def fill_gaps(z):
    """Interpolate NaN holes along rho. plot_surface renders NaN as a hole, and
    outer rho bins go sparse as rays shorten. Does not extrapolate past the
    ends — a bin with no data anywhere stays a hole, because that is real."""
    out = z.copy()
    for i in range(out.shape[0]):
        row, ok = out[i], ~np.isnan(out[i])
        if ok.sum() >= 2:
            idx = np.arange(row.size)
            out[i] = np.interp(idx, idx[ok], row[ok], left=np.nan, right=np.nan)
    return out


def polar_xy(THETA, RHO):
    """theta/rho grid -> Cartesian, so a surface sits over the nucleus footprint."""
    th = np.radians(THETA)
    return RHO * np.cos(th), RHO * np.sin(th)


print("helpers loaded")

## 4 · Nuclei table and the focal tracks

`Nucleus_ID` is the track id, so a nucleus keeps its identity across frames.
`FOCAL` is the four longest-lived tracks — the ones with enough timepoints to
show a trajectory rather than a point.

In [ ]:
df     = nsdb.nuclei(experiment=EXPERIMENT, qc="PASS")
df_all = nsdb.nuclei(experiment=EXPERIMENT, qc=None)
print(df.shape, "PASS  /", df_all.shape, "all")

span = (df_all.groupby("nucleus_id")
        .agg(n_frames=("time_frame", "nunique"),
             first=("time_frame", "min"), last=("time_frame", "max"))
        .sort_values(["n_frames", "first"], ascending=[False, True]))
FOCAL = span.head(N_FOCAL).index.tolist()
FC    = dict(zip(FOCAL, FOCAL_COLORS))

# Frames to analyse: those the focal tracks actually span.
FRAMES = sorted(df_all.loc[df_all.nucleus_id.isin(FOCAL), "time_frame"].unique())

print("\nfocal tracks:")
display(span.head(N_FOCAL))
print("frames:", FRAMES)

## 5 · Cross-sectional area over time

Population in grey, the four focal tracks in colour. `time_min` is the true
per-tile acquisition time, not `time_frame x 6` — nuclei in different tiles of
one frame were imaged up to 5 minutes apart.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(14, 5))

a.scatter(df.time_min, df.cross_sectional_area_um2, s=6, alpha=.15, color="0.5", lw=0)
med = df.groupby("time_frame").agg(t=("time_min", "median"),
                                   m=("cross_sectional_area_um2", "median"),
                                   lo=("cross_sectional_area_um2", lambda s: s.quantile(.25)),
                                   hi=("cross_sectional_area_um2", lambda s: s.quantile(.75)))
a.plot(med.t, med.m, "-", color="k", lw=2, label="population median")
a.fill_between(med.t, med.lo, med.hi, color="k", alpha=.10, lw=0)
for nid in FOCAL:
    g = df_all[df_all.nucleus_id == nid].sort_values("time_min")
    a.plot(g.time_min, g.cross_sectional_area_um2, "-o", ms=5, lw=1.6,
           color=FC[nid], label=short(nid))
a.set_xlabel("true acquisition time (min)")
a.set_ylabel("cross-sectional area (um^2)")
a.set_title("(a) Area over time — population and focal tracks")
a.legend(frameon=False, fontsize=8)

for nid in FOCAL:
    g = df_all[df_all.nucleus_id == nid].sort_values("time_min")
    if g.empty:
        continue
    b.plot(g.time_min, g.cross_sectional_area_um2 / g.cross_sectional_area_um2.iloc[0],
           "-o", ms=5, lw=1.6, color=FC[nid], label=short(nid))
b.axhline(1.0, color="k", lw=.7, ls="--")
b.set_xlabel("true acquisition time (min)")
b.set_ylabel("area / area at first frame")
b.set_title("(b) Fold change within each track")
b.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(FIGDIR / "area_timecourse.png", bbox_inches="tight")

### 5b · Binned distribution with the focal tracks overlaid

In [ ]:
edges = np.arange(0, df.time_min.max() + BIN_MIN, BIN_MIN)
labels = [f"{int(x)}-{int(y)}" for x, y in zip(edges[:-1], edges[1:])]
d = df.assign(bin=pd.cut(df.time_min, edges, labels=labels,
                         include_lowest=True)).dropna(subset=["bin"])
order = [c for c in labels if (d["bin"] == c).any()]
pos = {c: i + 1 for i, c in enumerate(order)}
data = [d.loc[d["bin"] == c, "cross_sectional_area_um2"].to_numpy() for c in order]

fig, ax = plt.subplots(figsize=(12, 5.5))
bp = boxplot(ax, data, order, patch_artist=True, showfliers=False,
             medianprops=dict(color="k"))
for box in bp["boxes"]:
    box.set(facecolor="#b8c6e8", alpha=.75, edgecolor="#33415c")
for i, v in enumerate(data, start=1):
    ax.scatter(np.random.normal(i, .07, v.size), v, s=4, color="k", alpha=.18, lw=0)
    ax.annotate(f"n={v.size}", (i, ax.get_ylim()[1]), ha="center", fontsize=8,
                color="0.35", xytext=(0, 4), textcoords="offset points")

for nid in FOCAL:
    g = df_all[df_all.nucleus_id == nid].copy()
    g["bin"] = pd.cut(g.time_min, edges, labels=labels, include_lowest=True)
    g = g.dropna(subset=["bin"]).sort_values("time_min")
    keep = [str(c) in pos for c in g["bin"]]
    ax.plot([pos[str(c)] for c, k in zip(g["bin"], keep) if k],
            g.loc[keep, "cross_sectional_area_um2"],
            "-o", ms=5, lw=1.6, color=FC[nid], label=short(nid), zorder=5)

ax.set_xlabel(f"acquisition time bin (min, width {int(BIN_MIN)})")
ax.set_ylabel("cross-sectional area (um^2)")
ax.set_title("Nuclear cross-sectional area per time interval, with focal tracks")
ax.legend(frameon=False, fontsize=8)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.savefig(FIGDIR / "area_bins.png", bbox_inches="tight")

## 6 · N/C ratio over time

From the mCherry halos with **no background subtraction** — the `background_*`
columns are empty in the export. Read the shape, not the absolute level.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(df.time_min, df.nc_ratio, s=6, alpha=.15, color="0.5", lw=0)
q = df.groupby("time_frame").agg(t=("time_min", "median"), m=("nc_ratio", "median"),
                                 lo=("nc_ratio", lambda s: s.quantile(.25)),
                                 hi=("nc_ratio", lambda s: s.quantile(.75)))
ax.plot(q.t, q.m, "-", color="k", lw=2, label="population median")
ax.fill_between(q.t, q.lo, q.hi, color="k", alpha=.10, lw=0)
for nid in FOCAL:
    g = df_all[df_all.nucleus_id == nid].sort_values("time_min")
    ax.plot(g.time_min, g.nc_ratio, "-o", ms=5, lw=1.6, color=FC[nid], label=short(nid))

ax.set_xlabel("true acquisition time (min)")
ax.set_ylabel("N/C ratio")
ax.set_title("N/C ratio over time  (no background subtraction)")
ax.legend(frameon=False, fontsize=8)
fig.savefig(FIGDIR / "nc_ratio.png", bbox_inches="tight")

## 7 · N/C ratio and area, dual axis

One panel per focal track, so the two quantities can be read against each other
within a single nucleus. The population panel comes first for reference.

In [ ]:
def dual(ax, t, nc, area, title, nc_band=None, area_band=None):
    """N/C on the left axis, area on the right, sharing a time axis."""
    ax.plot(t, nc, "-o", color="#14746f", ms=4, label="N/C ratio")
    if nc_band is not None:
        ax.fill_between(t, *nc_band, color="#14746f", alpha=.15, lw=0)
    ax.set_ylabel("N/C ratio", color="#14746f")
    ax.tick_params(axis="y", labelcolor="#14746f")
    ax.set_xlabel("time (min)")
    ax2 = ax.twinx()
    ax2.plot(t, area, "-s", color="#c1121f", ms=4, label="area")
    if area_band is not None:
        ax2.fill_between(t, *area_band, color="#c1121f", alpha=.13, lw=0)
    ax2.set_ylabel("area (um^2)", color="#c1121f")
    ax2.tick_params(axis="y", labelcolor="#c1121f")
    ax2.spines["right"].set_visible(True)
    ax.set_title(title, fontsize=10)
    return ax2


fig, axes = plt.subplots(1, N_FOCAL + 1, figsize=(4.6 * (N_FOCAL + 1), 4.4))

g = df.groupby("time_frame")
ax2 = dual(axes[0], g.time_min.median(), g.nc_ratio.median(),
           g.cross_sectional_area_um2.median(), "population (medians +/- IQR)",
           nc_band=(g.nc_ratio.quantile(.25), g.nc_ratio.quantile(.75)),
           area_band=(g.cross_sectional_area_um2.quantile(.25),
                      g.cross_sectional_area_um2.quantile(.75)))
h1, l1 = axes[0].get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
axes[0].legend(h1 + h2, l1 + l2, frameon=False, fontsize=8, loc="lower right")

for ax, nid in zip(axes[1:], FOCAL):
    t = df_all[df_all.nucleus_id == nid].sort_values("time_min")
    dual(ax, t.time_min, t.nc_ratio, t.cross_sectional_area_um2, short(nid))

fig.suptitle("N/C ratio (teal, left) and cross-sectional area (red, right)")
fig.tight_layout(rect=[0, 0, 1, .93])
fig.savefig(FIGDIR / "dual_axis.png", bbox_inches="tight")

## 8 · Load the radial sweep

Millions of rows — filter inside the parquet reader. The sweep ran on a single
channel (`cfg.radial_channel_index`), detected rather than assumed.

In [ ]:
CHANNEL = nsdb.radial(EXPERIMENT, columns=["channel"])["channel"].dropna().unique()
assert len(CHANNEL) == 1, f"expected one channel, got {CHANNEL}"
CHANNEL = CHANNEL[0]
print("channel:", CHANNEL)

sweep = nsdb.radial(
    EXPERIMENT,
    columns=["nucleus_id", "time_frame", "theta_deg", "rho_normalized",
             "channel", "intensity"],
    filters=[("channel", "==", CHANNEL), ("time_frame", "in", FRAMES)],
)
sweep = add_rho_wall_proxy(sweep)

missing = [n for n in FOCAL if n not in set(sweep.nucleus_id)]
if missing:
    print("WARNING: focal tracks absent from the sweep:", [short(m) for m in missing])
print(sweep.shape, "|", sweep.nucleus_id.nunique(), "nuclei")
sweep.head()

## 9 · Perinuclear rose — four tracks across time

Rows are the focal tracks, columns timepoints. Wedges are mean intensity in each
angular bin within the band just outside the nuclear surface; colour tracks the
same value. A lobe that persists across a row is a stable polarity; one that
moves is not.

`rho_wall_proxy` is reconstructed (see the helper docstring) — the band is
approximately, not exactly, the pipeline's nucleus-to-wall gap.

In [ ]:
GAP   = 0.30      # fraction of the nucleus-to-wall gap to include
NBINS = 24

band = sweep[(sweep.rho_wall_proxy > 0) & (sweep.rho_wall_proxy <= GAP)]

profiles, vmax = {}, 0.0
for nid in FOCAL:
    for t in FRAMES:
        sub = band[(band.nucleus_id == nid) & (band.time_frame == t)]
        if sub.empty:
            continue
        th, mean, _ = angular_profile(sub, NBINS)
        profiles[(nid, t)] = (th, np.nan_to_num(mean))
        vmax = max(vmax, float(np.nanmax(mean)))

fig, axes = plt.subplots(len(FOCAL), len(FRAMES),
                         figsize=(2.5 * len(FRAMES), 2.7 * len(FOCAL)),
                         subplot_kw={"projection": "polar"}, squeeze=False)
width = 2 * np.pi / NBINS
norm = plt.Normalize(0, vmax)
for r, nid in enumerate(FOCAL):
    for k, t in enumerate(FRAMES):
        ax = axes[r][k]; orient_polar(ax)
        ax.set_xticklabels([]); ax.set_yticklabels([])
        ax.set_ylim(0, vmax * 1.05)
        if (nid, t) not in profiles:
            ax.set_facecolor("0.97")
        else:
            th, vals = profiles[(nid, t)]
            ax.bar(th - width / 2, vals, width=width, align="edge",
                   color=plt.get_cmap("magma")(norm(vals)), edgecolor="none")
        if r == 0:
            ax.text(.5, 1.22, f"t = {t}", transform=ax.transAxes, ha="center", fontsize=9)
    axes[r][0].set_ylabel(short(nid), fontsize=9, labelpad=26)

fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="magma"), ax=axes,
             shrink=.5, label=f"mean {CHANNEL.lower()} intensity")
fig.suptitle(f"Perinuclear rose — within {GAP} of the nucleus-to-wall gap\n"
             "blank panels are frames where the track was not detected", fontsize=12)
fig.savefig(FIGDIR / "rose_perinuclear.png", bbox_inches="tight")

## 10 · Envelope rose — radius of peak intensity by angle

Four tracks, all timepoints overlaid by colour. The pipeline plots this in um;
only rho survives the export, so the shape is comparable and the scale is not.

In [ ]:
peaks = ray_peaks(sweep)
cmap = plt.get_cmap("viridis")
cn = {t: cmap(i / max(len(FRAMES) - 1, 1)) for i, t in enumerate(FRAMES)}

fig, axes = plt.subplots(1, len(FOCAL), figsize=(4.3 * len(FOCAL), 4.8),
                         subplot_kw={"projection": "polar"}, squeeze=False)
for ax, nid in zip(axes[0], FOCAL):
    orient_polar(ax)
    for t, g in peaks[peaks.nucleus_id == nid].groupby("time_frame"):
        g = g.sort_values("theta_deg")
        th, r = np.radians(g.theta_deg.to_numpy()), g.rho_at_peak.to_numpy()
        ax.plot(np.append(th, th[0]), np.append(r, r[0]), lw=.9, color=cn[t])
    ax.set_title(short(nid), fontsize=10)
    ax.set_ylim(0, 1)

fig.legend(handles=[plt.Line2D([], [], color=cn[t], label=f"t={t}") for t in FRAMES],
           loc="center right", frameon=False, fontsize=8, title="frame")
fig.suptitle("Radius of peak intensity by angle (rho, centre->wall) — focal tracks over time")
fig.tight_layout(rect=[0, 0, .93, .93])
fig.savefig(FIGDIR / "rose_individual.png", bbox_inches="tight")

### 10b · Pooled over all nuclei, per frame

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5), subplot_kw={"projection": "polar"})
orient_polar(ax)

for t in FRAMES:
    g = (peaks[peaks.time_frame == t].groupby("theta_deg")["rho_at_peak"]
         .agg(med="median", q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75))
         .reset_index().sort_values("theta_deg"))
    th = np.radians(g.theta_deg.to_numpy()); thc = np.append(th, th[0])
    ax.plot(thc, np.append(g["med"], g["med"].iloc[0]), lw=1.1, color=cn[t], label=f"t={t}")
    ax.fill_between(thc, np.append(g.q1, g.q1.iloc[0]),
                    np.append(g.q3, g.q3.iloc[0]), color=cn[t], alpha=.10, lw=0)

ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=8, title="frame", bbox_to_anchor=(1.22, 1.0))
ax.set_title("Radius of peak intensity by angle\nmedian +/- IQR, pooled over nuclei")
fig.savefig(FIGDIR / "rose_pooled.png", bbox_inches="tight")

## 11 · Intensity by angle and radius

Rows are the focal tracks, columns timepoints. A ring that stays at fixed rho
while the nucleus grows means the envelope is scaling with the nucleus.

In [ ]:
RHO_BINS = 60
d = sweep[sweep.nucleus_id.isin(FOCAL)]
edges = np.linspace(0, 1, RHO_BINS + 1)
vmin, vmax = np.nanpercentile(d.intensity, [2, 98])

fig, axes = plt.subplots(len(FOCAL), len(FRAMES),
                         figsize=(2.6 * len(FRAMES), 2.4 * len(FOCAL)), squeeze=False)
im = None
for r, nid in enumerate(FOCAL):
    for k, t in enumerate(FRAMES):
        ax = axes[r][k]
        sub = d[(d.nucleus_id == nid) & (d.time_frame == t)].copy()
        if sub.empty:
            ax.set_facecolor("0.97"); ax.set_xticks([]); ax.set_yticks([])
        else:
            sub["rb"] = pd.cut(sub.rho_normalized, edges, labels=False, include_lowest=True)
            grid = sub.pivot_table(index="theta_deg", columns="rb", values="intensity",
                                   aggfunc="mean").reindex(columns=range(RHO_BINS))
            im = ax.imshow(grid.to_numpy(), aspect="auto", origin="lower", cmap="inferno",
                           vmin=vmin, vmax=vmax, extent=[0, 1, 0, 360])
            if r == len(FOCAL) - 1:
                ax.set_xlabel("rho", fontsize=8)
            else:
                ax.set_xticklabels([])
            if k == 0:
                ax.set_yticks([0, 180, 360])
            else:
                ax.set_yticklabels([])
        if r == 0:
            ax.set_title(f"t={t}", fontsize=9)
        if k == 0:
            ax.set_ylabel(f"{short(nid)}\ntheta (deg)", fontsize=8)

if im is not None:
    fig.colorbar(im, ax=axes, shrink=.6, label=f"{CHANNEL} intensity (a.u.)")
fig.suptitle("Intensity by angle and radius (rho, centre->wall) — focal tracks over time")
fig.savefig(FIGDIR / "angle_distance.png", bbox_inches="tight")

## 12 · Membrane asymmetry

`R` is the intensity-weighted resultant after a per-shell baseline subtraction.
`R_geom` is the resultant of sample counts alone — what an isotropic nucleus
would show purely from droplet-wall clipping. **R must beat R_geom.**

In [ ]:
asym = asymmetry(sweep)
print(asym.shape)
display(asym.groupby("shell")[["R", "R_geom", "n_eff"]].median().round(3))

### 12a · Direction of the bright side over time

Grey bars are the population; coloured arrows are the focal tracks, so you can
see whether an individual nucleus holds its direction while the population stays
isotropic. Black line is the population resultant.

In [ ]:
SHELL = asym.shell.unique()[1]        # the shell holding the ring
NB = 24

d = asym[asym.shell == SHELL]
pop = population_direction(d).set_index("time_frame")
edges = np.linspace(0, 2 * np.pi, NB + 1)

fig, axes = plt.subplots(1, len(FRAMES), figsize=(3.1 * len(FRAMES), 4.0),
                         subplot_kw={"projection": "polar"}, squeeze=False)
for k, t in enumerate(FRAMES):
    ax = axes[0][k]; orient_polar(ax)
    phi = d.loc[d.time_frame == t, "phi"].dropna().to_numpy() % (2 * np.pi)
    counts, _ = np.histogram(phi, bins=edges)
    ax.bar(edges[:-1], counts, width=np.diff(edges), align="edge",
           color="0.82", edgecolor="w", lw=.4)
    hmax = counts.max() or 1
    if t in pop.index and np.isfinite(pop.loc[t, "R_pop"]):
        ax.plot([pop.loc[t, "mean_phi"]] * 2, [0, pop.loc[t, "R_pop"] * hmax],
                color="k", lw=1.8)
        ax.set_title(f"t={t}  n={phi.size}\nR={pop.loc[t,'R_pop']:.2f}, "
                     f"p={pop.loc[t,'p']:.3g}", fontsize=8)
    for nid in FOCAL:
        row = d[(d.nucleus_id == nid) & (d.time_frame == t)]
        if row.empty or not np.isfinite(row.phi.iloc[0]):
            continue
        ax.annotate("", xy=(row.phi.iloc[0], row.R.iloc[0] * hmax), xytext=(0, 0),
                    arrowprops=dict(color=FC[nid], width=1.4, headwidth=6, alpha=.9))
    ax.set_yticklabels([])

fig.legend(handles=[plt.Line2D([], [], color=FC[n], lw=2, label=short(n)) for n in FOCAL],
           loc="lower center", ncol=len(FOCAL), frameon=False, fontsize=8)
fig.suptitle(f"Direction of the bright side over time — shell rho {SHELL}\n"
             "grey = population, coloured arrows = focal tracks, black = population resultant")
fig.tight_layout(rect=[0, .07, 1, .86])
fig.savefig(FIGDIR / "direction_windrose.png", bbox_inches="tight")

### 12b · Asymmetry against time

In [ ]:
d = (asym[asym.shell == SHELL]
     .merge(df_all[["nucleus_id", "time_frame", "time_min"]],
            on=["nucleus_id", "time_frame"], how="left"))

fig, (a, b, c) = plt.subplots(1, 3, figsize=(15, 4.4))

for _, g in d.groupby("nucleus_id"):
    g = g.sort_values("time_min")
    a.plot(g.time_min, g.R, color="0.82", lw=.5, alpha=.7)
q = d.groupby("time_frame").agg(t=("time_min", "median"), m=("R", "median"),
                                lo=("R", lambda s: s.quantile(.25)),
                                hi=("R", lambda s: s.quantile(.75)))
a.plot(q.t, q.m, color="k", lw=2, label="population median +/- IQR")
a.fill_between(q.t, q.lo, q.hi, color="k", alpha=.12, lw=0)
for nid in FOCAL:
    g = d[d.nucleus_id == nid].sort_values("time_min")
    a.plot(g.time_min, g.R, "-o", ms=5, lw=1.6, color=FC[nid], label=short(nid))
a.set_xlabel("true acquisition time (min)"); a.set_ylabel("asymmetry score R")
a.set_title("(a) Asymmetry vs time"); a.legend(frameon=False, fontsize=8)

edges = np.arange(0, d.time_min.max() + BIN_MIN, BIN_MIN)
labels = [f"{int(x)}-{int(y)}" for x, y in zip(edges[:-1], edges[1:])]
dd = d.assign(bin=pd.cut(d.time_min, edges, labels=labels,
                         include_lowest=True)).dropna(subset=["bin"])
order = [x for x in labels if (dd["bin"] == x).any()]
bp = boxplot(b, [dd.loc[dd["bin"] == x, "R"].to_numpy() for x in order], order,
             patch_artist=True, showfliers=False, medianprops=dict(color="k"))
for box in bp["boxes"]:
    box.set(facecolor="#f4c6a8", alpha=.9, edgecolor="#8a5a3b")
b.set_xlabel(f"time bin (min, width {int(BIN_MIN)})"); b.set_ylabel("asymmetry score R")
b.set_title("(b) Distribution per time bin")
plt.setp(b.get_xticklabels(), rotation=45, ha="right")

for nid in FOCAL:
    g = d[d.nucleus_id == nid].sort_values("time_frame")
    if len(g) < 2:
        continue
    dphi = np.degrees(np.angle(np.exp(1j * np.diff(g.phi.to_numpy()))))
    c.plot(g.time_frame.to_numpy()[1:], np.abs(dphi), "-o", ms=5, color=FC[nid],
           label=short(nid))
c.axhline(90, color="k", ls="--", lw=.8, label="90 deg = uncorrelated")
c.set_xlabel("time frame"); c.set_ylabel("|change in phi| from previous frame (deg)")
c.set_ylim(0, 180); c.set_yticks([0, 45, 90, 135, 180])
c.set_title("(c) Does a track hold its direction?"); c.legend(frameon=False, fontsize=8)

fig.suptitle(f"Perinuclear membrane asymmetry — shell rho {SHELL}")
fig.tight_layout(rect=[0, 0, 1, .93])
fig.savefig(FIGDIR / "asymmetry_timecourse.png", bbox_inches="tight")

### 12c · By shell over time, against the geometry floor

In [ ]:
shells = sorted(asym.shell.unique())
cols = dict(zip(shells, SHELL_COLORS))

fig, (a, b, c) = plt.subplots(1, 3, figsize=(15, 4.4))
for s in shells:
    g = asym[asym.shell == s].groupby("time_frame")
    m = g.R.median()
    a.plot(m.index, m, "-o", color=cols[s], ms=4, label=f"rho {s}")
    a.fill_between(m.index, g.R.quantile(.25), g.R.quantile(.75),
                   color=cols[s], alpha=.15, lw=0)
    b.plot(m.index, m, "-o", color=cols[s], ms=4, label=f"rho {s}")
    b.plot(m.index, g.R_geom.median(), ":^", color=cols[s], ms=4)

a.axhline(np.sqrt(-np.log(.05) / max(int(asym.n_eff.median()), 1)), ls="--",
          color="k", lw=1, label="single-nucleus noise floor (p=0.05)")
a.set_xlabel("time frame"); a.set_ylabel("asymmetry score R")
a.set_title("(a) Asymmetry by shell over time"); a.legend(frameon=False, fontsize=7)
b.set_xlabel("time frame"); b.set_ylabel("R")
b.set_title("(b) Signal (solid) vs geometry floor (dotted)")

pop = population_direction(asym)
for s in shells:
    g = pop[pop.shell == s]
    c.plot(g.time_frame, g.R_pop, "-o", color=cols[s], ms=4, label=f"rho {s}")
    c.plot(g.time_frame, np.sqrt(-np.log(.05) / g.n.clip(lower=1)), "--",
           color=cols[s], lw=.8)
c.set_xlabel("time frame"); c.set_ylabel("population resultant of phi")
c.set_title("(c) Do nuclei agree on a lab direction?"); c.legend(frameon=False, fontsize=7)

fig.suptitle("Normalised asymmetry score by radial shell (rho, centre->wall)")
fig.tight_layout(rect=[0, 0, 1, .93])
fig.savefig(FIGDIR / "asymmetry_by_shell.png", bbox_inches="tight")

### 12d · Aggregate, direction discarded

In [ ]:
NB = 36
fig, (a, b, c) = plt.subplots(1, 3, figsize=(15, 4.4))

data = [asym.loc[asym.shell == s, "R"].dropna().to_numpy() for s in shells]
parts = a.violinplot(data, showmedians=True, showextrema=True)
for pc, s in zip(parts["bodies"], shells):
    pc.set_facecolor(cols[s]); pc.set_alpha(.55)
for i, s in enumerate(shells, start=1):
    a.plot([i - .35, i + .35], [asym.loc[asym.shell == s, "R_geom"].median()] * 2,
           ":", color="#c1121f", lw=1.6)
a.set_xticks(range(1, len(shells) + 1)); a.set_xticklabels(shells, fontsize=7)
a.set_xlabel("shell (rho, centre->wall)"); a.set_ylabel("asymmetry score R")
a.plot([], [], ":", color="#c1121f", label="median R_geom")
a.set_title("(a) Score distribution, all frames"); a.legend(frameon=False, fontsize=8)

for s in shells:
    g = asym[asym.shell == s]
    for col, ls, lab in (("R", "-", f"rho {s}"), ("R_geom", ":", None)):
        v = np.sort(g[col].dropna().to_numpy())
        if v.size:
            b.plot(v, np.arange(1, v.size + 1) / v.size, ls, color=cols[s], lw=1.3, label=lab)
b.set_xlabel("asymmetry score R"); b.set_ylabel("cumulative fraction")
b.set_title("(b) ECDF (dotted = R_geom)"); b.legend(frameon=False, fontsize=7)

phi_lookup = asym.set_index(["nucleus_id", "time_frame", "shell"])["phi"]
centres = np.linspace(-180, 180, NB, endpoint=False) + 180 / NB
for s in shells:
    lo, hi = asym.loc[asym.shell == s, ["shell_lo", "shell_hi"]].iloc[0]
    sub_all = sweep[(sweep.rho_normalized > lo) & (sweep.rho_normalized <= hi)]
    stack = []
    for (nid, t), g in sub_all.groupby(["nucleus_id", "time_frame"], sort=False):
        phi = phi_lookup.get((nid, t, s), np.nan)
        if not np.isfinite(phi):
            continue
        _, mean, _ = angular_profile(g, NB)
        if not np.isfinite(mean).any() or np.nanmean(mean) <= 0:
            continue
        shift = int(round(np.degrees(phi) / (360 / NB)))
        stack.append(np.roll(mean / np.nanmean(mean), -shift + NB // 2))
    if stack:
        arr = np.vstack(stack)
        med = np.nanmedian(arr, axis=0)
        se = np.nanstd(arr, axis=0) / max(np.sqrt(arr.shape[0]), 1)
        c.plot(centres, med, color=cols[s], lw=1.4, label=f"rho {s} (n={arr.shape[0]})")
        c.fill_between(centres, med - se, med + se, color=cols[s], alpha=.2, lw=0)
c.axhline(1.0, color="k", lw=.7)
c.set_xlabel("angle from each nucleus's own asymmetry axis (deg)")
c.set_ylabel("intensity / shell mean"); c.set_xticks([-180, -90, 0, 90, 180])
c.set_title("(c) Alignment-averaged profile"); c.legend(frameon=False, fontsize=7)

fig.suptitle("Aggregate membrane asymmetry — direction discarded")
fig.tight_layout(rect=[0, 0, 1, .93])
fig.savefig(FIGDIR / "asymmetry_aggregate.png", bbox_inches="tight")

### 12e · Shell x frame rose grid, one figure per focal track

Wedge radius is sqrt(intensity above baseline), so wedge **area** is proportional
to intensity. Grey wedges are angles with no sample — almost always droplet-wall
clipping. Red arrow is the resultant.

In [ ]:
NB = 36
edges = np.linspace(0, 2 * np.pi, NB + 1)

for nid in FOCAL:
    d = sweep[sweep.nucleus_id == nid]
    fig, axes = plt.subplots(len(SHELLS), len(FRAMES),
                             figsize=(2.2 * len(FRAMES), 2.4 * len(SHELLS)),
                             subplot_kw={"projection": "polar"}, squeeze=False)
    for r, ((lo, hi), col) in enumerate(zip(SHELLS, SHELL_COLORS)):
        for k, t in enumerate(FRAMES):
            ax = axes[r][k]; orient_polar(ax)
            ax.set_xticklabels([]); ax.set_yticklabels([])
            sub = d[(d.time_frame == t) & (d.rho_normalized > lo)
                    & (d.rho_normalized <= hi)]
            if sub.empty:
                ax.set_facecolor("0.97")
            else:
                th, mean, count = angular_profile(sub, NB)
                base = np.nanpercentile(mean[np.isfinite(mean)], 10)
                w = np.clip(np.nan_to_num(mean - base), 0, None)
                h = np.sqrt(w); hmax = h.max() or 1
                miss = count == 0
                if miss.any():
                    ax.bar(edges[:-1][miss], np.full(miss.sum(), hmax),
                           width=np.diff(edges)[miss], align="edge",
                           color="0.88", zorder=0)
                ax.bar(edges[:-1], h, width=np.diff(edges), align="edge",
                       color=col, edgecolor="w", lw=.3)
                R, phi = resultant(th, w)
                if np.isfinite(R):
                    ax.annotate("", xy=(phi, R * hmax), xytext=(0, 0),
                                arrowprops=dict(color="#c1121f", width=1.2, headwidth=6))
                    ax.set_title(f"R={R:.2f}", fontsize=7, color="0.3")
            if k == 0:
                ax.set_ylabel(f"rho ({lo:.2f}, {hi:.2f}]", fontsize=7)
            if r == 0:
                ax.text(.5, 1.30, f"t = {t}", transform=ax.transAxes,
                        ha="center", fontsize=9)
    fig.suptitle(f"Intensity by angle and shell over time — {short(nid)}\n"
                 "wedge area prop. to intensity above baseline · red = resultant · "
                 "grey = no sample", fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, .90])
    fig.savefig(FIGDIR / f"shell_rose_{short(nid)}.png", bbox_inches="tight")
    plt.show()

## 13 · Intensity surfaces over time

Rows are the focal tracks, columns timepoints. rho and theta are mapped to
Cartesian, so each surface sits over the nucleus's own footprint with the
nucleus at the centre and Z as intensity. Z is on a shared scale so panels are
comparable across both nuclei and time.

In [ ]:
RHO_BINS = 60
d = sweep[sweep.nucleus_id.isin(FOCAL)]
zmin, zmax = np.nanpercentile(d.intensity, [1, 99])

fig = plt.figure(figsize=(3.0 * len(FRAMES), 3.0 * len(FOCAL)))
for r, nid in enumerate(FOCAL):
    for k, t in enumerate(FRAMES):
        ax = fig.add_subplot(len(FOCAL), len(FRAMES), r * len(FRAMES) + k + 1,
                             projection="3d")
        one = d[(d.nucleus_id == nid) & (d.time_frame == t)]
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
        if one.empty:
            ax.set_axis_off()
        else:
            THETA, RHO, Z = sweep_grid(one, rho_bins=RHO_BINS)
            X, Y = polar_xy(THETA, RHO)
            ax.plot_surface(X, Y, fill_gaps(Z), cmap="inferno", linewidth=0,
                            antialiased=True, vmin=zmin, vmax=zmax)
            ax.set_zlim(zmin, zmax)
            ax.view_init(elev=45, azim=-58)
        if r == 0:
            ax.set_title(f"t = {t}", fontsize=9)
        if k == 0:
            ax.text2D(-0.15, 0.5, short(nid), transform=ax.transAxes,
                      rotation=90, va="center", fontsize=9)

fig.suptitle(f"{CHANNEL} intensity surfaces — focal tracks over time "
             f"(shared Z scale {zmin:.0f}-{zmax:.0f})", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, .95])
fig.savefig(FIGDIR / "surface_focal_grid.png", bbox_inches="tight")

### 13b · One track, large enough to read the envelope structure

In [ ]:
NID, T = FOCAL[0], FRAMES[-1]
one = sweep[(sweep.nucleus_id == NID) & (sweep.time_frame == T)]

THETA, RHO, Z = sweep_grid(one, rho_bins=RHO_BINS)
Z = fill_gaps(Z)
X, Y = polar_xy(THETA, RHO)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
s = ax.plot_surface(X, Y, Z, cmap="inferno", linewidth=0, antialiased=True,
                    rstride=1, cstride=1)
ax.set_xlabel("x (rho)"); ax.set_ylabel("y (rho)"); ax.set_zlabel("intensity")
ax.view_init(elev=42, azim=-58)
fig.colorbar(s, shrink=.55, label=f"{CHANNEL} intensity (a.u.)")
ax.set_title(f"{CHANNEL} intensity surface — {short(NID)}, t={T}")
fig.savefig(FIGDIR / "surface_polar.png", bbox_inches="tight")

### 13c · Pooled across nuclei, over time

Drop the focal filter and the pivot's mean pools everything — shows whether a
feature is systematic or belongs to one nucleus.

In [ ]:
fig = plt.figure(figsize=(3.2 * len(FRAMES), 3.6))
for k, t in enumerate(FRAMES):
    ax = fig.add_subplot(1, len(FRAMES), k + 1, projection="3d")
    allnuc = sweep[sweep.time_frame == t]
    THETA_a, RHO_a, Z_a = sweep_grid(allnuc, rho_bins=RHO_BINS)
    X_a, Y_a = polar_xy(THETA_a, RHO_a)
    ax.plot_surface(X_a, Y_a, fill_gaps(Z_a), cmap="viridis", linewidth=0,
                    antialiased=True, vmin=zmin, vmax=zmax)
    ax.set_zlim(zmin, zmax)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.view_init(elev=45, azim=-58)
    ax.set_title(f"t = {t}  (n={allnuc.nucleus_id.nunique()})", fontsize=9)

fig.suptitle("Pooled intensity surface over time — shared Z scale", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, .90])
fig.savefig(FIGDIR / "surface_pooled.png", bbox_inches="tight")

## 14 · Figures written

In [ ]:
for p in sorted(FIGDIR.glob("*.png")):
    print(f"{p.stat().st_size/1e3:8.0f} kB  {p.name}")